In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget

import logging
from core.Log import *
import pandas as pd
from core.benchmarks import *


C:\Users\sulei\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [1]:
INNER_CV_parameters = load_from_json("NCV_5_3_folds/OUTER_experiments.json")
df = pd.DataFrame(INNER_CV_parameters)
df


NameError: name 'load_from_json' is not defined

In [8]:

def setup_loggers5KBENCHMARKS():

	INNERtrain = logging.getLogger('INNER_5KtrainBENCHMARKS')
	INNERtrain.setLevel(logging.INFO)

	INNERtrain.propagate = False

	if not INNERtrain.hasHandlers():
		INNERtrain.propagate = False
		train_handler = logging.FileHandler('INNER_5KtrainBENCHMARKS.log', mode='a')		#REMEMBER: 'a' mode to append logs
		train_handler.setFormatter(logging.Formatter('%(asctime)s ;		 %(message)s', datefmt='%H:%M'))
		INNERtrain.addHandler(train_handler)

setup_loggers5KBENCHMARKS()



In [9]:
import pickle
from torch.utils.data import Dataset, DataLoader

class DataLoaderFactoryMLP:
	"""
	Creates PyTorch DataLoaders for specific folds of a pre-computed
	cross-validation setup.
	"""
	def __init__(self, main_dataset):
		self.main_dataset = main_dataset
		#self.all_folds_data = all_folds_data
		# You can also store constants like num_workers here
		self.num_workers = 4

	def create_inner_loaders(self, outer_fold_id, inner_fold_id):
		"""
		Generates train and validation dataloaders for a specific inner fold.
		"""
		with open("NCV_5_3_folds/folds_indices_stats.pkl", "rb") as f:
			all_folds_data = pickle.load(f)
		# --- 1. Get the correct data for the specified fold ---
		outer_fold_struct = all_folds_data[outer_fold_id]
		inner_fold_struct = outer_fold_struct['INNER_FOLDS'][inner_fold_id]

		# The outer_train_pool is the dataset from which inner folds are made
		outer_train_pool = [self.main_dataset[i] for i in outer_fold_struct['OUTER_FOLD_TRAIN_idx']]

		# Get the specific train/val data for the inner fold
		# Note: The indices are local to the outer_train_pool
		train_fold_data = [outer_train_pool[i] for i in inner_fold_struct['INNER_FOLD_TRAIN_idx']]
		val_fold_data = [outer_train_pool[i] for i in inner_fold_struct['INNER_FOLD_VAL_idx']]

		inner_stats = inner_fold_struct['INNER_FOLD_stats']


		# --- 3. Create Dataset and DataLoader objects ---
		train_dataset = METADataset(train_fold_data, inner_stats)
		val_dataset = METADataset(val_fold_data, inner_stats)

		train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=self.num_workers)
		val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=self.num_workers)

		return train_loader, val_loader



def get_fold_stats(outer_fold_id=None, inner_fold_id=None):
	"""
	Returns a tuple containing
		(outer_fold_stats, inner_fold_stats)
	for the specified fold.
	"""
	with open("NCV_5_3_folds/folds_indices_stats.pkl", "rb") as f:
		all_folds_data = pickle.load(f)
	if outer_fold_id is None and inner_fold_id is None:
		return all_folds_data
	elif outer_fold_id is None:
		# Return all outer folds stats
		return [fold['OUTER_FOLD_stats'] for fold in all_folds_data]
	elif inner_fold_id is None:
		# Return all inner folds stats for the specified outer fold
		return [fold['INNER_FOLD_stats'] for fold in all_folds_data[outer_fold_id]['INNER_FOLDS']]
	else:
		# Return stats for the specified outer and inner fold
		outer_fold_data = all_folds_data[outer_fold_id]
		inner_fold_data = outer_fold_data['INNER_FOLDS'][inner_fold_id]
		return outer_fold_data['OUTER_FOLD_stats'], inner_fold_data['INNER_FOLD_stats']




class METADataset(Dataset):
	def __init__(self, data_dicts, stats):
		self.data_dicts = data_dicts
		self.stats = stats

	def __len__(self):
		return len(self.data_dicts)

	def __getitem__(self, idx):
		case_dict = self.data_dicts[idx]

		# One-hot encode gender ('F' -> [1, 0], 'M' -> [0, 1])
		gender = [1.0, 0.0] if case_dict['gender'] == 'F' else [0.0, 1.0]
		age = (case_dict['age'] - self.stats['AGEmean']) / self.stats['AGEstd']

		meta = torch.tensor([age] + gender, dtype=torch.float32)
		label = torch.tensor(case_dict['label'], dtype=torch.float32)

		return {"CaseID": case_dict['ID'],
			"label": label, "meta": meta}





In [10]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from sklearn.metrics import roc_auc_score, roc_curve, f1_score
import numpy as np

def append_experiment_results(item, path="NCV_5_3_folds/INNER_resultsBENCHMARKS.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")

def best_threshold(all_labels, all_probs, utility="youden"):
	"""
	Pick a post-hoc decision threshold on validation predictions.
	utility: "youden" (maximize TPR-FPR) or "f1".
	"""
	if utility == "f1":
		# scan unique probabilities for F1
		# (for speed you can sample a subset if very large)
		thr = np.unique(all_probs)
		f1s = [f1_score(all_labels, all_probs >= t) for t in thr]
		idx = int(np.argmax(f1s))
		return float(thr[idx])
	else:
		fpr, tpr, thr = roc_curve(all_labels, all_probs)
		j = tpr - fpr
		idx = int(np.argmax(j))
		return float(thr[idx])  # may be outside [0,1] if degenerate; fine.



def train_INNER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('INNER_5Ktrain')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	hypers = experiment['hypers']
	ExpID = experiment['ExpID']
	epochs = hypers['Epochs']

	LR = hypers['LR']
	WD = hypers['WD']
	P = hypers['P']

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=2, factor=0.5,
								  threshold=1e-3, threshold_mode='rel',
								  cooldown=0, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	best_auc = -np.inf
	best_loss_at_best_auc = np.inf
	best_th = 0.5

	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)


	print(f"	↳ Experiment {ExpID} | Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0

		for batch in train_loader:

			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)

		T_loss = running_loss / train_N

		model.eval()
		running_loss = 0.0
		all_labels = []  #y_true
		all_probs = []   #y_pred

		with torch.no_grad():
			for batch in val_loader:
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				all_labels.extend(lbl.cpu())
				all_probs.extend(torch.sigmoid(logits).cpu())

		V_loss = running_loss / val_N
		all_labels = torch.cat(all_labels).numpy().reshape(-1)
		all_probs = torch.cat(all_probs).numpy().reshape(-1)
		val_auc = roc_auc_score(all_labels, all_probs)

		scheduler.step(V_loss)
		th_star = best_threshold(all_labels, all_probs)
		improved = val_auc > best_auc + 1e-6

		log.info(f"{experiment['Model']};    {ExpID};    {experiment['OUTER_FOLD']};    {experiment['INNER_FOLD']};    {hypers['HPset']};    {epoch:02d};    {T_loss:.4f};    {V_loss:.4f}    {val_auc:.4f};    {th_star:.6f};    {optimizer.param_groups[0]['lr']};    {no_improve:02d};")

		if improved:
			best_auc = val_auc
			best_loss_at_best_auc = V_loss
			best_th = th_star
			best_epoch = epoch
			no_improve = 0
		else:
			no_improve += 1
		if no_improve >= 3: break

	results = {
		"ExpID": ExpID,
		"Model": experiment['Model'],
		"best_val_auc": float(best_auc),
		"best_val_loss": float(best_loss_at_best_auc),
		"best_threshold": float(best_th),
		"best_epoch": int(best_epoch),
		"epochs_ran": int(epoch)}

	return results


In [11]:
full_dl = load_dataset_info(file="NCV_5_3_folds/data_info_5-3NCV.json")
main_dataset = [i for i in full_dl if i["pool"] == 'main']

DLMLP = DataLoaderFactoryMLP(main_dataset)

INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "META+MLP"
	and exp["OUTER_FOLD"] == 0       # 2, 3, 4
	#and exp["hypers"]["HPset"] == 4
	and exp["trained"] == False
]

print(f"experiments to do: {len(filtered)}")

CompletedExps = []

log = logging.getLogger('INNER_5KtrainBENCHMARKS')
log.info("18:25 ; Model; ExpID; OUTER_FOLD; INNER_FOLD; HPset; epoch;  T_loss;    V_loss;    val_auc;    th_star;    LR;    no_improve;")
for experiment in filtered:
	print(experiment)
	ExpID = experiment['ExpID']
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']
	train_loader, val_loader = DLMLP.create_inner_loaders(OUT, INN)
	DR = experiment['hypers']['DR']
	model = MetadataMLP(DR)
	results = train_INNER_model(model, train_loader, val_loader, experiment)
	append_experiment_results(results, path="NCV_5_3_folds/INNER_resultsBENCHMARKS.jsonl")
	CompletedExps.append(experiment['ExpID'])



Loaded NCV_5_3_folds/INNER_experiments.json.
experiments to do: 12
{'ExpID': 61, 'Model': 'META+MLP', 'OUTER_FOLD': 0, 'INNER_FOLD': 0, 'hypers': {'HPset': 1, 'LR': 0.0003, 'WD': 1e-06, 'DR': 0.4, 'P': 5, 'Epochs': 30}, 'trained': False}
	↳ Experiment 61 | Training model... 


RuntimeError: DataLoader worker (pid(s) 3060, 35636, 19712, 34328) exited unexpectedly